In [ ]:
#@title Convert XML file to CSV file (SKIP THIS CELL)

import xml.etree.ElementTree as ET
import csv

tree = ET.parse('Data/OpenStreetMap Trace for a Sparse Traffic.xml')  # replace with your file path
root = tree.getroot()

# open a file for writing
with open('Data/sparse.csv', 'w', newline='') as out_csv:
    writer = csv.writer(out_csv)
    writer.writerow(["time", "id", "x", "y", "angle", "type", "speed", "pos", "lane", "slope"])

    for timestep in root.findall('timestep'):
        time = timestep.get('time')
        for vehicle in timestep.findall('vehicle'):
            id = vehicle.get('id')
            x = vehicle.get('x')
            y = vehicle.get('y')
            angle = vehicle.get('angle')
            type = vehicle.get('type')
            speed = vehicle.get('speed')
            pos = vehicle.get('pos')
            lane = vehicle.get('lane')
            slope = vehicle.get('slope')
            writer.writerow([time, id, x, y, angle, type, speed, pos, lane, slope])

In [ ]:
#@title Read the input data (Decompress sparse.zip if you haven't already) - (SKIP THIS CELL IF YOU ARE WORKING WITH preprocessed_sparse.csv)

import pandas as pd

# from google.colab import drive
# drive.mount('/content/drive')

# Load data
data = pd.read_csv('Data/sparse.csv')
# data = pd.read_csv('/content/drive/MyDrive/2024-05- 7088CEM (ANN Module)/Assignments/handover_prediction/Data/sparse.csv')
data.head()
data.info()
data.describe()

In [ ]:
#@title Preprocess data (getting rid of unnecessary columns) (SKIP THIS CELL IF YOU ARE WORKING WITH preprocessed_sparse.csv (You can decompress the zip file with the same name))
# Convert the 'time' column to float and then to integer
data['time'] = data['time'].astype(float).astype(int)

# Remove 'veh' from 'id' column and convert to integer
data['id'] = data['id'].str.replace('veh', '').astype(int)

# Remove specified columns
columns_to_remove = ['type', 'lane', 'slope', 'pos']
data = data.drop(columns=columns_to_remove)

# Display the first few rows of the processed data
data.head(20)
data.info()
data.describe()

# Save the preprocessed DataFrame to a CSV file for future use
# output_path = '/content/drive/MyDrive/2024-05-7088CEM (ANN Module)/Assignments/handover_prediction/Data/preprocessed_sparse.csv'
output_path = 'Data/preprocessed_sparse.csv'
data.to_csv(output_path, index=False)

print(f"Preprocessed data saved to {output_path}")

In [7]:
#@title Preprocesing: Preparing train and test datasets (Save sequenced data) (YOU CAN SKIP THIS BY DECOMPRESSING sequences_vehicle_ids.zip)
import pickle
import pandas as pd

save_path = f'Data/preprocessed_sparse.csv'
data = pd.read_csv(save_path)

sequence_length = 5

def create_sequences(data, sequence_length):
    sequences = [] # each seqence has the properties of one vehicle in different time steps (sequential)
    vehicle_ids = []
    for veh_id, group in data.groupby('id'):
        group = group.sort_values(by='time')
        for i in range(len(group) - sequence_length):
            seq = group.iloc[i:i+sequence_length]
            target = group.iloc[i+sequence_length][['x', 'y']].values  # Use the next row's coordinates as target
            sequences.append((seq[['x', 'y', 'speed', 'angle']].values, target))
            vehicle_ids.append(veh_id)
    return sequences, vehicle_ids

sequences, vehicle_ids = create_sequences(data, sequence_length)
# Save sequences and vehicle IDs to a file

# Print three example sequences and their targets
for i in range(3):
    print(f"\n---\nSequence {i+1}:")
    print("Features (x, y, speed, angle):")
    print(sequences[i][0])
    print("Target (next x, y):")
    print(sequences[i][1])

# with open('/content/drive/MyDrive/2024-05- 7088CEM (ANN Module)/Assignments/handover_prediction/Data/sequences_vehicle_ids.pkl', 'wb') as f:
with open('Data/sequences_vehicle_ids.pkl', 'wb') as f:
    pickle.dump((sequences, vehicle_ids), f)

# how to load it later?
# with open('Data/sequences_vehicle_ids.pkl', 'rb') as f:
#     sequences, vehicle_ids = pickle.load(f)


---
Sequence 1:
Features (x, y, speed, angle):
[[5.01420e+02 1.47312e+03 0.00000e+00 2.90000e-01]
 [5.01430e+02 1.47508e+03 1.96000e+00 2.90000e-01]
 [5.01370e+02 1.47955e+03 4.44000e+00 3.59340e+02]
 [5.01110e+02 1.48568e+03 6.02000e+00 3.57640e+02]
 [5.00760e+02 1.49423e+03 8.40000e+00 3.57640e+02]]
Target (next x, y):
[ 500.34 1504.42]

---
Sequence 2:
Features (x, y, speed, angle):
[[5.01430e+02 1.47508e+03 1.96000e+00 2.90000e-01]
 [5.01370e+02 1.47955e+03 4.44000e+00 3.59340e+02]
 [5.01110e+02 1.48568e+03 6.02000e+00 3.57640e+02]
 [5.00760e+02 1.49423e+03 8.40000e+00 3.57640e+02]
 [5.00340e+02 1.50442e+03 1.00100e+01 3.57640e+02]]
Target (next x, y):
[ 499.83 1516.71]

---
Sequence 3:
Features (x, y, speed, angle):
[[ 501.37 1479.55    4.44  359.34]
 [ 501.11 1485.68    6.02  357.64]
 [ 500.76 1494.23    8.4   357.64]
 [ 500.34 1504.42   10.01  357.64]
 [ 499.83 1516.71   12.26  357.61]]
Target (next x, y):
[ 499.24 1530.83]
